# 04 — CNNs preentrenadas en Rust: ResNet18, VGG19 y MobileNetV2

**Objetivo:** descargar checkpoints **reales** compatibles con [tch-rs](https://github.com/LaurentMazare/tch-rs), comprobar que existen y preparar estrategias de *transfer learning*, *fine-tuning* y *deep fine-tuning*. **No entrenamos** ni afirmamos haber entrenado ninguno.

Este notebook usa el kernel **Rust / Evcxr**. Las celdas ejecutadas usan la biblioteca estándar de Rust y `curl`, evitando exigir libtorch en el *easy setup*. El ejemplo Rust con **tch/libtorch que carga tensores y cambia `requires_grad` de verdad** está en [`examples/cnn_finetuning/src/main.rs`](../examples/cnn_finetuning/src/main.rs), para ejecución opcional en una máquina con libtorch. **Los checks de este notebook no equivalen a cargar pesos en tch ni a ejecutar inferencia.**

⚠️ **Descargas grandes:** ResNet18 ≈47 MB, MobileNetV2 ≈14 MB y VGG19 ≈575 MB. Se guardan en un directorio temporal fuera de Git. Necesitas Internet, `curl`, espacio en disco y aceptar las condiciones/licencias aplicables de los pesos. Las URLs corresponden a los assets de [tch-rs / release mw](https://github.com/LaurentMazare/tch-rs/releases/tag/mw).

## 1. Catálogo de checkpoints compatibles con Rust

No confundas arquitecturas con pesos: **`.ot`** son pesos exportados a formato cargable por `tch::nn::VarStore::load`; los `.pth` de torchvision no se importan directamente como `.ot`.

In [ ]:
let checkpoints: [(&str, &str, u64); 3] = [
    ("ResNet18", "resnet18.ot", 46_831_783),
    ("MobileNetV2", "mobilenet-v2.ot", 14_249_059),
    ("VGG19", "vgg19.ot", 574_678_008),
];
for (name, file, bytes) in checkpoints {
    println!("{name:<12} {file:<18} {bytes:>10} bytes ({:.1} MiB)", bytes as f64 / 1_048_576.0);
}
println!("Fuente: https://github.com/LaurentMazare/tch-rs/releases/tag/mw");

## 2. Descargar los tres checkpoints desde Rust

Descarga efectiva con `std::process::Command` y `curl --fail --location`, con comprobación de tamaño para detectar descargas truncadas. Los archivos **no se incorporan al repositorio**. Una segunda ejecución reutiliza el caché, si el tamaño coincide. En GitHub Actions los pesos se descargan en el runner efímero y solo se guardan los *outputs de texto*.

In [ ]:
let cache_dir = std::env::temp_dir().join("rust-for-ai-cnn-weights");
std::fs::create_dir_all(&cache_dir).expect("Cannot create model cache directory");
let checkpoints: [(&str, &str, u64); 3] = [
    ("ResNet18", "resnet18.ot", 46_831_783),
    ("MobileNetV2", "mobilenet-v2.ot", 14_249_059),
    ("VGG19", "vgg19.ot", 574_678_008),
];
if std::env::var("RUST_FOR_AI_CNN_OFFLINE").as_deref() == Ok("1") {
    println!("Offline Rust syntax check: downloads skipped (CI compile/run smoke test).");
} else {
for (name, file, expected) in checkpoints {
    let destination = cache_dir.join(file);
    let found = std::fs::metadata(&destination).map(|x| x.len()).unwrap_or(0);
    if found != expected {
        let temp = destination.with_extension("ot.part");
        let url = format!("https://github.com/LaurentMazare/tch-rs/releases/download/mw/{file}");
        let status = std::process::Command::new("curl")
            .args(["--fail", "--location", "--silent", "--show-error", "--retry", "2", "--connect-timeout", "30", "--max-time", "600", "--output"])
            .arg(&temp).arg(&url).status().expect("curl must be installed");
        assert!(status.success(), "Download failed for {name}: {url}");
        let actual = std::fs::metadata(&temp).expect("Missing download").len();
        assert_eq!(actual, expected, "Truncated checkpoint: {name}");
        std::fs::rename(&temp, &destination).expect("Cannot move downloaded model");
    }
    let actual = std::fs::metadata(&destination).unwrap().len();
    assert_eq!(actual, expected);
    println!("OK: {name} checkpoint descargado/verificado ({actual} bytes)");
}
println!("Checkpoints descargados: 3/3; todavía NO se han cargado como tensores.");
}

## 3. Tres estrategias de ajuste en Rust

**Linear probing / head-only:** congelar toda la CNN y entrenar **solo una cabeza nueva**. **Deep fine-tuning:** descongelar bloques finales (o parte del backbone) además de la cabeza. **Full fine-tuning:** descongelar todos los parámetros, normalmente con una tasa de aprendizaje menor.

El archivo [`examples/cnn_finetuning/src/main.rs`](../examples/cnn_finetuning/src/main.rs) implementa `VarStore::load`, `VarStore::freeze`, `Tensor::set_requires_grad`, `VarStore::unfreeze` y una capa `nn::linear` de salida en Rust. **Importante:** la demostración usa una cabeza adaptadora sobre los *1000 logits ImageNet* del modelo original para compartir un ejemplo entre las tres arquitecturas; no equivale a sustituir su clasificador final por una cabeza alimentada directamente por *features*. Para transfer learning de alto rendimiento, suele preferirse usar el extractor de características y sustituir el clasificador original.

La ejecución de esa variante requiere **tch + libtorch/PyTorch compatible**, que no forman parte del setup básico, y no se ejecuta automáticamente por ser considerablemente más pesado.

In [ ]:
let modes = [
    ("head", "backbone congelado + nueva cabeza de clasificación"),
    ("deep", "cabeza y último bloque del backbone entrenables"),
    ("full", "todos los parámetros entrenables"),
];
for (mode, detail) in modes {
    println!("{mode:<5}: {detail}");
}
println!("Ver código tch ejecutable: examples/cnn_finetuning/src/main.rs");
println!("Entrenamiento realizado en este notebook: NINGUNO.");

## 4. Cómo probar la manipulación real de parámetros con `tch` (opcional)

Después de ejecutar el notebook (ya tendrás los pesos en el caché temporal), instala una versión de **PyTorch/libtorch compatible con la versión fijada de `tch`** y ejecuta desde la raíz del repo:

```bash
# Instala libtorch/PyTorch compatible con tch según https://github.com/LaurentMazare/tch-rs
# Por ejemplo, si usas un entorno PyTorch compatible:
export LIBTORCH_USE_PYTORCH=1
cargo run --manifest-path examples/cnn_finetuning/Cargo.toml -- resnet18 head 3
cargo run --manifest-path examples/cnn_finetuning/Cargo.toml -- mobilenet-v2 deep 3
cargo run --manifest-path examples/cnn_finetuning/Cargo.toml -- vgg19 full 3
```

Los comandos construyen la arquitectura, cargan los pesos `.ot`, crean una cabeza adaptadora de **3 clases** y cambian los indicadores `requires_grad`. **No ejecutan una época de entrenamiento ni actualizan pesos**. El ejecutable lee los pesos del mismo caché temporal (en otra máquina debes ejecutar la descarga del notebook allí).

Para entrenar después, necesitarás imágenes etiquetadas, preprocesamiento ImageNet (RGB, escala y normalización), una función de pérdida y un optimizador. No midas *accuracy* antes de entrenar y evaluar en datos reales.